In [2]:

# ============================================================
# MEMBER 3: CREDIT CARD FRAUD DETECTION
# Model Training and Evaluation
# ============================================================

import os
import gc
import json
import time
import warnings
import joblib

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20

# Project paths
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "creditcard.csv"
MEMBER2_DIR = BASE_DIR / "member2_output"
ARTIFACT_DIR = MEMBER2_DIR / "recovered_artifacts"
OUTPUT_DIR = BASE_DIR / "member3_output"

OUTPUT_DIR.mkdir(exist_ok=True)

# Model configuration
TARGET = "Class"
N_JOBS = 2

print("Libraries imported successfully.")
print("Project directory:", BASE_DIR)
print("Output directory:", OUTPUT_DIR)


Libraries imported successfully.
Project directory: c:\Users\m7129\Desktop\credit-card-fraud-detection
Output directory: c:\Users\m7129\Desktop\credit-card-fraud-detection\member3_output


In [3]:

# ============================================================
# 2. LOAD AND VERIFY MEMBER 2 DATA
# ============================================================

# Load existing Member 2 exports
train_parts = [
    pd.read_csv(MEMBER2_DIR / f"member2_train_balanced_part{i}.csv")
    for i in range(1, 4)
]

train_balanced = pd.concat(train_parts, ignore_index=True)
test_export = pd.read_csv(MEMBER2_DIR / "member2_test.csv")

# Load original dataset and reproduce Member 2's cleaning and split
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=[TARGET]).drop_duplicates().copy()
df[TARGET] = df[TARGET].astype("int8")

feature_columns = [c for c in df.columns if c != TARGET]

X = df[feature_columns]
y = df[TARGET]

X_dev, X_test_raw, y_dev, y_test_raw = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# Verify exported test data against reconstructed original test split
assert len(test_export) == len(X_test_raw), "Test row count mismatch!"
assert np.array_equal(
    y_test_raw.to_numpy(),
    test_export[TARGET].to_numpy()
), "Test labels or order mismatch!"

assert np.allclose(
    X_test_raw.to_numpy(),
    X_test_raw.to_numpy()
), "Original test data check failed!"

assert list(test_export.columns) == feature_columns + [TARGET]

# Verify balanced training exports
assert list(train_balanced.columns) == feature_columns + [TARGET]
assert train_balanced[TARGET].value_counts().nunique() == 1

print("DATA VERIFICATION PASSED")
print("-" * 45)

print("Original cleaned dataset:", df.shape)
print("Development partition:", X_dev.shape)
print("Untouched test partition:", X_test_raw.shape)

print("\nBalanced training export:", train_balanced.shape)
print("Balanced class counts:")
print(train_balanced[TARGET].value_counts().sort_index())

print("\nUntouched test class counts:")
print(test_export[TARGET].value_counts().sort_index())

print("\nTest labels and row order match Member 2 export.")
print("Feature columns match.")


DATA VERIFICATION PASSED
---------------------------------------------
Original cleaned dataset: (283726, 31)
Development partition: (226980, 30)
Untouched test partition: (56746, 30)

Balanced training export: (453204, 31)
Balanced class counts:
Class
0    226602
1    226602
Name: count, dtype: int64

Untouched test class counts:
Class
0    56651
1       95
Name: count, dtype: int64

Test labels and row order match Member 2 export.
Feature columns match.


In [4]:

# ============================================================
# 3. LEAKAGE-FREE VALIDATION PREPARATION
# ============================================================

# Split original development data into inner training and validation
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_dev
)

# Fit preprocessing ONLY on inner training data
validation_imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True
)

X_train_imputed = validation_imputer.fit_transform(X_train_raw)
X_val_imputed = validation_imputer.transform(X_val_raw)

validation_scaler = RobustScaler()

X_train_scaled = validation_scaler.fit_transform(X_train_imputed)
X_val_scaled = validation_scaler.transform(X_val_imputed)

# Reduce memory consumption
X_train_scaled = X_train_scaled.astype(np.float32)
X_val_scaled = X_val_scaled.astype(np.float32)

# Apply SMOTE ONLY to inner training data
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)

X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train_scaled,
    y_train
)

y_train_balanced = np.asarray(y_train_balanced)
y_val = np.asarray(y_val)

print("VALIDATION DATA PREPARATION COMPLETE")
print("-" * 45)

print("Inner training before SMOTE:", X_train_scaled.shape)
print("Inner training after SMOTE:", X_train_balanced.shape)
print("Validation data:", X_val_scaled.shape)

print("\nTraining class distribution after SMOTE:")
print(pd.Series(y_train_balanced).value_counts().sort_index())

print("\nValidation class distribution (untouched):")
print(pd.Series(y_val).value_counts().sort_index())

assert X_train_balanced.shape[0] == len(y_train_balanced)
assert X_val_scaled.shape[0] == len(y_val)
assert len(np.unique(y_val)) == 2

print("\nNo validation samples were used for SMOTE or fitting preprocessing.")


VALIDATION DATA PREPARATION COMPLETE
---------------------------------------------
Inner training before SMOTE: (181584, 30)
Inner training after SMOTE: (362564, 30)
Validation data: (45396, 30)

Training class distribution after SMOTE:
0    181282
1    181282
Name: count, dtype: int64

Validation class distribution (untouched):
0    45320
1       76
Name: count, dtype: int64

No validation samples were used for SMOTE or fitting preprocessing.


In [5]:

# ============================================================
# 4. MODEL 1: LOGISTIC REGRESSION BASELINE
# ============================================================

from sklearn.exceptions import ConvergenceWarning

model_results = {}
model_objects = {}
validation_scores = {}

logistic_model = LogisticRegression(
    solver="lbfgs",
    C=0.1,
    max_iter=300,
    class_weight=None,
    random_state=RANDOM_STATE
)

print("Training Logistic Regression...")
start_time = time.time()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    logistic_model.fit(X_train_balanced, y_train_balanced)

training_time = time.time() - start_time

# Obtain fraud probabilities on untouched validation data
val_scores = logistic_model.predict_proba(X_val_scaled)[:, 1]
val_predictions = (val_scores >= 0.5).astype("int8")

# Evaluate
ap = average_precision_score(y_val, val_scores)
roc = roc_auc_score(y_val, val_scores)
precision = precision_score(y_val, val_predictions, zero_division=0)
recall = recall_score(y_val, val_predictions, zero_division=0)
f1 = f1_score(y_val, val_predictions, zero_division=0)
cm = confusion_matrix(y_val, val_predictions)

model_objects["Logistic Regression"] = logistic_model
validation_scores["Logistic Regression"] = val_scores

model_results["Logistic Regression"] = {
    "Average Precision": ap,
    "ROC-AUC": roc,
    "Precision@0.5": precision,
    "Recall@0.5": recall,
    "F1@0.5": f1,
    "Training Time (seconds)": training_time
}

print("\nLOGISTIC REGRESSION RESULTS")
print("-" * 45)
print(f"Training time: {training_time:.2f} seconds")
print(f"Average Precision: {ap:.5f}")
print(f"ROC-AUC: {roc:.5f}")
print(f"Precision at 0.5: {precision:.5f}")
print(f"Recall at 0.5: {recall:.5f}")
print(f"F1 at 0.5: {f1:.5f}")

print("\nConfusion Matrix:")
print(cm)

if any(issubclass(w.category, ConvergenceWarning) for w in caught):
    print("\nWARNING: Logistic Regression did not converge.")
    print("We will review convergence before accepting this baseline.")
else:
    print("\nConvergence check: Passed.")


Training Logistic Regression...

LOGISTIC REGRESSION RESULTS
---------------------------------------------
Training time: 2.99 seconds
Average Precision: 0.79040
ROC-AUC: 0.97151
Precision at 0.5: 0.05163
Recall at 0.5: 0.89474
F1 at 0.5: 0.09763

Confusion Matrix:
[[44071  1249]
 [    8    68]]

Convergence check: Passed.


In [6]:

# ============================================================
# 5. MODEL 2: LIGHTGBM
# ============================================================

lightgbm_model = lgb.LGBMClassifier(
    n_estimators=200,
    num_leaves=15,
    max_depth=5,
    learning_rate=0.05,
    min_child_samples=50,
    max_bin=127,
    random_state=RANDOM_STATE,
    n_jobs=2,
    verbosity=-1
)

print("Training LightGBM...")
start_time = time.time()

lightgbm_model.fit(
    X_train_balanced,
    y_train_balanced,
    eval_set=[(X_val_scaled, y_val)],
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(25, verbose=False),
        lgb.log_evaluation(0)
    ]
)

training_time = time.time() - start_time

# Validation fraud probabilities
val_scores = lightgbm_model.predict_proba(X_val_scaled)[:, 1]
val_predictions = (val_scores >= 0.5).astype("int8")

# Evaluation
ap = average_precision_score(y_val, val_scores)
roc = roc_auc_score(y_val, val_scores)
precision = precision_score(y_val, val_predictions, zero_division=0)
recall = recall_score(y_val, val_predictions, zero_division=0)
f1 = f1_score(y_val, val_predictions, zero_division=0)
cm = confusion_matrix(y_val, val_predictions)

model_objects["LightGBM"] = lightgbm_model
validation_scores["LightGBM"] = val_scores

model_results["LightGBM"] = {
    "Average Precision": ap,
    "ROC-AUC": roc,
    "Precision@0.5": precision,
    "Recall@0.5": recall,
    "F1@0.5": f1,
    "Training Time (seconds)": training_time
}

print("\nLIGHTGBM RESULTS")
print("-" * 45)
print(f"Training time: {training_time:.2f} seconds")
print("Best iteration:", lightgbm_model.best_iteration_)
print(f"Average Precision: {ap:.5f}")
print(f"ROC-AUC: {roc:.5f}")
print(f"Precision at 0.5: {precision:.5f}")
print(f"Recall at 0.5: {recall:.5f}")
print(f"F1 at 0.5: {f1:.5f}")
print("\nConfusion Matrix:")
print(cm)


Training LightGBM...

LIGHTGBM RESULTS
---------------------------------------------
Training time: 4.15 seconds
Best iteration: 38
Average Precision: 0.82088
ROC-AUC: 0.97006
Precision at 0.5: 0.09770
Recall at 0.5: 0.89474
F1 at 0.5: 0.17617

Confusion Matrix:
[[44692   628]
 [    8    68]]


In [7]:

# ============================================================
# MODEL 3: XGBOOST
# ============================================================

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    max_bin=128,
    n_jobs=2,
    random_state=RANDOM_STATE,
    eval_metric="aucpr",
    early_stopping_rounds=25
)

print("Training XGBoost...")
start_time = time.time()

xgb_model.fit(
    X_train_balanced,
    y_train_balanced,
    eval_set=[(X_val_scaled, y_val)],
    verbose=False
)

training_time = time.time() - start_time

val_scores = xgb_model.predict_proba(X_val_scaled)[:, 1]
val_predictions = (val_scores >= 0.5).astype("int8")

ap = average_precision_score(y_val, val_scores)
roc = roc_auc_score(y_val, val_scores)
precision = precision_score(y_val, val_predictions, zero_division=0)
recall = recall_score(y_val, val_predictions, zero_division=0)
f1 = f1_score(y_val, val_predictions, zero_division=0)
cm = confusion_matrix(y_val, val_predictions)

model_objects["XGBoost"] = xgb_model
validation_scores["XGBoost"] = val_scores

model_results["XGBoost"] = {
    "Average Precision": ap,
    "ROC-AUC": roc,
    "Precision@0.5": precision,
    "Recall@0.5": recall,
    "F1@0.5": f1,
    "Training Time (seconds)": training_time
}

print("\nXGBOOST RESULTS")
print("-" * 45)
print(f"Training time: {training_time:.2f} seconds")
print(f"Best iteration: {xgb_model.best_iteration}")
print(f"Average Precision: {ap:.5f}")
print(f"ROC-AUC: {roc:.5f}")
print(f"Precision at 0.5: {precision:.5f}")
print(f"Recall at 0.5: {recall:.5f}")
print(f"F1 at 0.5: {f1:.5f}")
print("\nConfusion Matrix:")
print(cm)


Training XGBoost...

XGBOOST RESULTS
---------------------------------------------
Training time: 5.21 seconds
Best iteration: 29
Average Precision: 0.81828
ROC-AUC: 0.96339
Precision at 0.5: 0.11221
Recall at 0.5: 0.89474
F1 at 0.5: 0.19941

Confusion Matrix:
[[44782   538]
 [    8    68]]


In [7]:

# ============================================================
# MODEL 4: RANDOM FOREST
# ============================================================

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=5,
    max_features="sqrt",
    max_samples=0.7,
    n_jobs=2,
    random_state=RANDOM_STATE,
    class_weight=None
)

print("Training Random Forest...")
start_time = time.time()

rf_model.fit(X_train_balanced, y_train_balanced)

training_time = time.time() - start_time

val_scores = rf_model.predict_proba(X_val_scaled)[:, 1]
val_predictions = (val_scores >= 0.5).astype("int8")

ap = average_precision_score(y_val, val_scores)
roc = roc_auc_score(y_val, val_scores)
precision = precision_score(y_val, val_predictions, zero_division=0)
recall = recall_score(y_val, val_predictions, zero_division=0)
f1 = f1_score(y_val, val_predictions, zero_division=0)
cm = confusion_matrix(y_val, val_predictions)

model_objects["Random Forest"] = rf_model
validation_scores["Random Forest"] = val_scores

model_results["Random Forest"] = {
    "Average Precision": ap,
    "ROC-AUC": roc,
    "Precision@0.5": precision,
    "Recall@0.5": recall,
    "F1@0.5": f1,
    "Training Time (seconds)": training_time
}

print("\nRANDOM FOREST RESULTS")
print("-" * 45)
print(f"Training time: {training_time:.2f} seconds")
print(f"Average Precision: {ap:.5f}")
print(f"ROC-AUC: {roc:.5f}")
print(f"Precision at 0.5: {precision:.5f}")
print(f"Recall at 0.5: {recall:.5f}")
print(f"F1 at 0.5: {f1:.5f}")
print("\nConfusion Matrix:")
print(cm)


Training Random Forest...


NameError: name 'X_train_balanced' is not defined

In [2]:
from sklearn.ensemble import RandomForestClassifier

print("Random Forest imported successfully.")


Random Forest imported successfully.


In [4]:
RANDOM_STATE = 42
N_JOBS = 2

print("Configuration restored.")


Configuration restored.


In [6]:
import time
import gc
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

RANDOM_STATE = 42
N_JOBS = 2

print("Required imports and configuration restored.")

Required imports and configuration restored.


In [8]:

# NOTEBOOK STATE DIAGNOSTIC

required_objects = [
    "RandomForestClassifier",
    "RANDOM_STATE",
    "time",
    "np",
    "pd",
    "X_train_balanced",
    "y_train_balanced",
    "X_val_scaled",
    "y_val",
    "model_results",
    "model_objects",
    "validation_scores",
    "LogisticRegression",
    "lgb",
    "xgb"
]

print("NOTEBOOK STATE AUDIT")
print("-" * 45)

for name in required_objects:
    exists = name in globals()
    print(f"{name:25} : {'AVAILABLE' if exists else 'MISSING'}")

print("-" * 45)

if "model_results" in globals():
    print("Completed models:", list(model_results.keys()))

if "model_objects" in globals():
    print("Stored model objects:", list(model_objects.keys()))

if "validation_scores" in globals():
    print("Stored validation scores:", list(validation_scores.keys()))



NOTEBOOK STATE AUDIT
---------------------------------------------
RandomForestClassifier    : AVAILABLE
RANDOM_STATE              : AVAILABLE
time                      : AVAILABLE
np                        : AVAILABLE
pd                        : AVAILABLE
X_train_balanced          : MISSING
y_train_balanced          : MISSING
X_val_scaled              : MISSING
y_val                     : MISSING
model_results             : MISSING
model_objects             : MISSING
validation_scores         : MISSING
LogisticRegression        : MISSING
lgb                       : MISSING
xgb                       : MISSING
---------------------------------------------
